# Two-record hop, no filler, Kaggle T4

Same Tesla T4. Do not switch GPUs. Same Qwen3-4B revision, FP16, greedy decoding.

No filler in any condition. Same question, choices, answer instructions, and 48-token answer allowance.

**Cell 4** reruns the same 10 `hop_dev` examples into `/kaggle/working/results_stage_hop_nofiller`. It does **not** resume `results_stage_hop`.

**Cell 6** runs the 40 `hop_test` examples (240 evals, no no-message control). Run it only if the 10-example checklist is healthy: iterative retrieval still finds both records, oracle text remains reliable, and there is no implementation problem. Do not keep tuning until KV wins.

Do not load `data/test.jsonl`.

In [ ]:
import os, json, shutil, sys
from pathlib import Path

os.environ["LATENT_RELAY_INFER_DTYPE"] = "fp16"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

ON_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()

candidates = [Path.cwd()]
if Path("/kaggle/input").is_dir():
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if d not in {".git", ".pytest_cache"}]
        if "stage_hop.py" in files and Path(root).name == "src":
            candidates.append(Path(root).parent)

REPO = None
for cand in candidates:
    if (cand / "src" / "stage_hop.py").is_file() and (cand / "data" / "hop_dev.jsonl").is_file():
        REPO = cand
        break
if REPO is None:
    raise FileNotFoundError("Need src/stage_hop.py and data/hop_dev.jsonl. Re-upload the repo zip.")

if ON_KAGGLE:
    dest = WORK / "latent-relay-audit"
    if str(REPO).startswith("/kaggle/input"):
        shutil.copytree(REPO, dest, dirs_exist_ok=True)
    elif not (dest / "src" / "stage_hop.py").is_file():
        shutil.copytree(REPO, dest, dirs_exist_ok=True)
    REPO = dest
    os.chdir(REPO)
else:
    os.chdir(REPO)

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("REPO", REPO.resolve())
print("hop_dev", sum(1 for _ in (REPO / "data" / "hop_dev.jsonl").open(encoding="utf-8")))
print("hop_test", sum(1 for _ in (REPO / "data" / "hop_test.jsonl").open(encoding="utf-8")))

In [ ]:
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.57.3", "tokenizers==0.22.2", "huggingface_hub==0.36.2",
    "safetensors==0.7.0", "accelerate==1.13.0", "datasets==4.4.1",
])
import torch, transformers
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print("cuda", torch.cuda.is_available(), gpu)
print("transformers", transformers.__version__)
assert transformers.__version__.startswith("4.57")
if gpu is not None and "T4" not in gpu:
    raise RuntimeError(f"Stay on the locked Tesla T4. Got {gpu}.")

In [ ]:
subprocess.check_call([sys.executable, "scripts/fetch_upstream.py"])
print("Dev dest /kaggle/working/results_stage_hop_nofiller (does not resume results_stage_hop)")
print("Test dest /kaggle/working/results_stage_hop_test")

In [ ]:
rc = subprocess.call([sys.executable, "-m", "src.stage_hop", "--split", "hop_dev"])
print("hop_dev nofiller exit", rc)

In [ ]:
src = WORK / "results_stage_hop_nofiller"
if not (src / "summary.json").is_file():
    src = REPO / "results" / "stage_hop_nofiller" / "run"
print("reading", src)
for name in ("summary.json", "manifest.json"):
    path = src / name
    if path.is_file():
        print("====", name)
        print(path.read_text(encoding="utf-8")[:8000])

# 40-example hop_test — run only if the 10-example no-filler checklist is healthy

Healthy means: iterative retrieval still finds both records, oracle text remains reliable, and there is no implementation problem. Do not keep tuning until latent communication wins.

Download `/kaggle/working/results_stage_hop_test` during the run.

In [ ]:
rc = subprocess.call([sys.executable, "-m", "src.stage_hop", "--split", "hop_test"])
print("hop_test exit", rc)

In [ ]:
src = WORK / "results_stage_hop_test"
if not (src / "summary.json").is_file():
    src = REPO / "results" / "stage_hop_test" / "run"
print("reading", src)
for name in ("summary.json", "manifest.json"):
    path = src / name
    if path.is_file():
        print("====", name)
        print(path.read_text(encoding="utf-8")[:8000])